Install libraries

In [ ]:
!pip install datasets spacy pandas
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Load Enron dataset from HuggingFace

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("corbt/enron-emails")

df = pd.DataFrame(dataset['train'])

print("Columns:", df.columns)
print("Rows:", len(df))
df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/185M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/141M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/517401 [00:00<?, ? examples/s]

Columns: Index(['message_id', 'subject', 'from', 'to', 'cc', 'bcc', 'date', 'body',
       'file_name'],
      dtype='object')
Rows: 517401


,message_id,subject,from,to,cc,bcc,date,body,file_name
0,<18782981.1075855378110.JavaMail.evans@thyme>,,phillip.allen@enron.com,[tim.belden@enron.com],[],[],2001-05-14 23:39:00+00:00,Here is our forecast\n\n,allen-p/_sent_mail/1.
1,<15464986.1075855378456.JavaMail.evans@thyme>,Re:,phillip.allen@enron.com,[john.lavorato@enron.com],[],[],2001-05-04 20:51:00+00:00,Traveling to have a business meeting takes the...,allen-p/_sent_mail/10.
2,<24216240.1075855687451.JavaMail.evans@thyme>,Re: test,phillip.allen@enron.com,[leah.arsdall@enron.com],[],[],2000-10-18 10:00:00+00:00,test successful. way to go!!!,allen-p/_sent_mail/100.
3,<13505866.1075863688222.JavaMail.evans@thyme>,,phillip.allen@enron.com,[randall.gay@enron.com],[],[],2000-10-23 13:13:00+00:00,"Randy,\n\n Can you send me a schedule of the s...",allen-p/_sent_mail/1000.
4,<30922949.1075863688243.JavaMail.evans@thyme>,Re: Hello,phillip.allen@enron.com,[greg.piper@enron.com],[],[],2000-08-31 12:07:00+00:00,Let's shoot for Tuesday at 11:45.,allen-p/_sent_mail/1001.


Cleaning function

In [ ]:
import re

def clean_email(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
print(df.columns)


Index(['message_id', 'subject', 'from', 'to', 'cc', 'bcc', 'date', 'body',
       'file_name'],
      dtype='object')


Apply cleaning

In [ ]:
df['clean_message'] = df['body'].apply(clean_email)
df[['body', 'clean_message']].head()


,body,clean_message
0,Here is our forecast\n\n,Here is our forecast
1,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...
2,test successful. way to go!!!,test successful. way to go!!!
3,"Randy,\n\n Can you send me a schedule of the s...","Randy, Can you send me a schedule of the salar..."
4,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.


Transformation (word count)

In [ ]:
df['word_count'] = df['clean_message'].apply(lambda x: len(x.split()))
df[['clean_message', 'word_count']].head()
#df['word_count'] = df['clean_message'].apply(lambda x: len(x.split()))


,clean_message,word_count
0,Here is our forecast,4
1,Traveling to have a business meeting takes the...,139
2,test successful. way to go!!!,5
3,"Randy, Can you send me a schedule of the salar...",34
4,Let's shoot for Tuesday at 11:45.,6


Enrichment (NER)

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text[:1000])
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    return str(ents)

# Apply NER only on a small sample
sample_df = df.head(2000).copy()

sample_df['entities'] = sample_df['clean_message'].apply(extract_entities)

# Create empty column for all rows
df['entities'] = ""

# Fill entities only for sample rows
df.loc[sample_df.index, 'entities'] = sample_df['entities']

df[['clean_message', 'entities']].head()

#for entire dataset enrichment
#df['entities'] = df['clean_message'].apply(extract_entities)
#df[['clean_message', 'entities']].head()


,clean_message,entities
0,Here is our forecast,[]
1,Traveling to have a business meeting takes the...,"[('Austin', 'PERSON')]"
2,test successful. way to go!!!,[]
3,"Randy, Can you send me a schedule of the salar...","[('Randy', 'PERSON'), ('Patti', 'PERSON'), ('P..."
4,Let's shoot for Tuesday at 11:45.,"[('Tuesday', 'DATE'), ('11:45', 'CARDINAL')]"


Normalization (final dataset)

In [ ]:
df = df[['message_id', 'from', 'to', 'subject', 'date', 'body',
         'clean_message', 'word_count', 'entities']]
df.head()

#df = df[['body', 'clean_message', 'word_count', 'entities']]

#df = df[['from', 'to', 'subject', 'date', 'body',
       #  'clean_message', 'word_count', 'entities']]

,message_id,from,to,subject,date,body,clean_message,word_count,entities
0,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,[tim.belden@enron.com],,2001-05-14 23:39:00+00:00,Here is our forecast\n\n,Here is our forecast,4,[]
1,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,[john.lavorato@enron.com],Re:,2001-05-04 20:51:00+00:00,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...,139,"[('Austin', 'PERSON')]"
2,<24216240.1075855687451.JavaMail.evans@thyme>,phillip.allen@enron.com,[leah.arsdall@enron.com],Re: test,2000-10-18 10:00:00+00:00,test successful. way to go!!!,test successful. way to go!!!,5,[]
3,<13505866.1075863688222.JavaMail.evans@thyme>,phillip.allen@enron.com,[randall.gay@enron.com],,2000-10-23 13:13:00+00:00,"Randy,\n\n Can you send me a schedule of the s...","Randy, Can you send me a schedule of the salar...",34,"[('Randy', 'PERSON'), ('Patti', 'PERSON'), ('P..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,phillip.allen@enron.com,[greg.piper@enron.com],Re: Hello,2000-08-31 12:07:00+00:00,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.,6,"[('Tuesday', 'DATE'), ('11:45', 'CARDINAL')]"


Save CSV

In [ ]:
df.to_csv("cleaned_enron_emails.csv", index=False)
print("✅ File saved")


✅ File saved


Verify saved file

In [ ]:
check_df = pd.read_csv("cleaned_enron_emails.csv")
print(len(check_df))
check_df.head()


/tmp/ipykernel_915/4228037128.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  check_df = pd.read_csv("cleaned_enron_emails.csv")


517401


,message_id,from,to,subject,date,body,clean_message,word_count,entities
0,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,['tim.belden@enron.com'],NaN,2001-05-14 23:39:00+00:00,Here is our forecast\n\n,Here is our forecast,4,[]
1,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,['john.lavorato@enron.com'],Re:,2001-05-04 20:51:00+00:00,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...,139,"[('Austin', 'PERSON')]"
2,<24216240.1075855687451.JavaMail.evans@thyme>,phillip.allen@enron.com,['leah.arsdall@enron.com'],Re: test,2000-10-18 10:00:00+00:00,test successful. way to go!!!,test successful. way to go!!!,5,[]
3,<13505866.1075863688222.JavaMail.evans@thyme>,phillip.allen@enron.com,['randall.gay@enron.com'],NaN,2000-10-23 13:13:00+00:00,"Randy,\n\n Can you send me a schedule of the s...","Randy, Can you send me a schedule of the salar...",34,"[('Randy', 'PERSON'), ('Patti', 'PERSON'), ('P..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,phillip.allen@enron.com,['greg.piper@enron.com'],Re: Hello,2000-08-31 12:07:00+00:00,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.,6,"[('Tuesday', 'DATE'), ('11:45', 'CARDINAL')]"




Download(day2)

In [ ]:
from google.colab import files
files.download("cleaned_enron_emails.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Save ORIGINAL dataset from HuggingFace to CSV

In [ ]:
df[['body']].to_csv("original_enron_emails.csv", index=False)


Download it:

In [ ]:
from google.colab import files
files.download("original_enron_emails.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download CLEANED dataset

In [ ]:
#files.download("cleaned_enron_emails.csv")


Create script files directly in Colab (very smart trick)

In [ ]:
# load_data.py
with open("load_data.py", "w") as f:
    f.write("""
from datasets import load_dataset
import pandas as pd

def load_data():
    dataset = load_dataset("corbt/enron-emails")
    df = pd.DataFrame(dataset['train'])
    return df
""")

# clean_data.py
with open("clean_data.py", "w") as f:
    f.write("""
import re

def clean_email(text):
    text = str(text)
    text = re.sub(r'\\n', ' ', text)
    text = re.sub(r'\\S+@\\S+', '', text)
    text = re.sub(r'http\\S+', '', text)
    text = re.sub(r'[^A-Za-z ]', '', text)
    return text.lower()

def apply_cleaning(df):
    df['clean_message'] = df['message'].apply(clean_email)
    return df
""")

# transform.py
with open("transform.py", "w") as f:
    f.write("""
def transform_data(df):
    df['word_count'] = df['clean_message'].apply(lambda x: len(x.split()))
    return df
""")

# enrich.py
with open("enrich.py", "w") as f:
    f.write("""
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text[:1000])
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    return str(ents)

def enrich_data(df):
    df['entities'] = df['clean_message'].apply(extract_entities)
    return df
""")

# main.py
with open("main.py", "w") as f:
    f.write("""
from load_data import load_data
from clean_data import apply_cleaning
from transform import transform_data
from enrich import enrich_data

df = load_data()
df = apply_cleaning(df)
df = transform_data(df)
df = enrich_data(df)

df = df[['message', 'clean_message', 'word_count', 'entities']]
df.to_csv("cleaned_enron_emails.csv", index=False)

print("Milestone 1 completed")
""")

print("✅ Script files created")


✅ Script files created


Generate Structured Triples

In [ ]:
import pandas as pd
import ast
import numpy as np # Import numpy for np.ndarray

triples = []

for _, row in df.iterrows():

    email_id = f"Email_{row['message_id']}"
    sender = row['from']

    # --- Communication Triples ---
    if pd.notna(sender): # 'from' is typically a single email string
        triples.append((sender, "SENT", email_id))

    # Handle 'to' recipients (which can be lists, NaN, or occasionally other types)
    receiver_data = row['to']
    current_recipients = []

    if receiver_data is None: # Explicitly check for Python None
        current_recipients = []
    elif isinstance(receiver_data, (list, np.ndarray, pd.Series)): # Handle list, array, or Series of recipients first
        # If it's already a list/array/series, convert to list and filter NaNs
        current_recipients = [str(item) for item in receiver_data if pd.notna(item) and str(item).strip()]
    elif pd.isna(receiver_data): # Now check for pandas NaN (only if not a list-like already)
        current_recipients = []
    elif isinstance(receiver_data, str):
        # If string, try literal_eval, else treat as single string
        try:
            # Handle empty string for entities explicitly
            if receiver_data.strip() == '':
                current_recipients = []
            else:
                temp_list = ast.literal_eval(receiver_data)
                if isinstance(temp_list, list):
                    current_recipients = [str(item) for item in temp_list if pd.notna(item) and str(item).strip()]
                else:
                    current_recipients = [str(temp_list)] if pd.notna(temp_list) and str(temp_list).strip() else []
        except (ValueError, SyntaxError):
            # If literal_eval fails, assume it's a single email string.
            current_recipients = [receiver_data] if receiver_data.strip() else [] # Check for empty string
    else: # Fallback for any other non-list, non-string, non-None scalar value
        current_recipients = [str(receiver_data)] if str(receiver_data).strip() else [] # Ensure it's not empty after stripping

    # Now process the current_recipients list
    for r_email in current_recipients:
        if r_email: # Only add if recipient string is not empty
            triples.append((email_id, "SENT_TO", r_email))

    # --- Topic Triple (from subject) ---
    if pd.notna(row['subject']):
        triples.append((email_id, "ABOUT", row['subject']))

    # --- Entity Triples (from NER column) ---
    if 'entities' in df.columns and pd.notna(row['entities']):
        entities_data = row['entities']
        entities_list = []

        if isinstance(entities_data, str):
            if entities_data.strip() == "": # Handle empty string case explicitly
                entities_list = []
            else:
                try:
                    entities_list = ast.literal_eval(entities_data)
                    if not isinstance(entities_list, list): # Ensure it's a list, otherwise treat as empty
                        entities_list = []
                except (ValueError, SyntaxError):
                    entities_list = [] # If parsing fails, treat as empty
        elif isinstance(entities_data, list):
            entities_list = entities_data

        for entity, label in entities_list:
            if pd.notna(entity): # Ensure entity is not NaN
                triples.append((email_id, "MENTIONS", str(entity)))


# Convert to DataFrame
triples_df = pd.DataFrame(triples, columns=["Subject", "Relation", "Object"])

triples_df.head()

,Subject,Relation,Object
0,phillip.allen@enron.com,SENT,Email_<18782981.1075855378110.JavaMail.evans@t...
1,Email_<18782981.1075855378110.JavaMail.evans@t...,SENT_TO,tim.belden@enron.com
2,Email_<18782981.1075855378110.JavaMail.evans@t...,ABOUT,
3,phillip.allen@enron.com,SENT,Email_<15464986.1075855378456.JavaMail.evans@t...
4,Email_<15464986.1075855378456.JavaMail.evans@t...,SENT_TO,john.lavorato@enron.com


In [ ]:
# Check unique values and value counts for the 'subject' column
print("Unique subject values (sample):")
display(df['subject'].value_counts(dropna=False).head(10))

# Check for empty strings in 'subject'
empty_subject_count = df['subject'].apply(lambda x: isinstance(x, str) and x.strip() == '').sum()
print(f"\nNumber of rows with empty string subjects: {empty_subject_count}")

# Display some rows where 'subject' is NaN or an empty string
print("\nSample rows with NaN or empty string subjects:")
display(df[df['subject'].isna() | (df['subject'].apply(lambda x: isinstance(x, str) and x.strip() == ''))].head())

Unique subject values (sample):


,count
subject,
,19187
RE:,6477
Re:,6306
Demand Ken Lay Donate Proceeds from Enron Stock Sales,1124
FW:,938
Schedule Crawler: HourAhead Failure,900
Schedule Crawler: HourAhead Failure <CODESITE>,800
Enron Mentions,784
EnTouch Newsletter,518



Number of rows with empty string subjects: 19187

Sample rows with NaN or empty string subjects:


,message_id,from,to,subject,date,body,clean_message,word_count,entities
0,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,[tim.belden@enron.com],,2001-05-14 23:39:00+00:00,Here is our forecast\n\n,Here is our forecast,4,[]
3,<13505866.1075863688222.JavaMail.evans@thyme>,phillip.allen@enron.com,[randall.gay@enron.com],,2000-10-23 13:13:00+00:00,"Randy,\n\n Can you send me a schedule of the s...","Randy, Can you send me a schedule of the salar...",34,"[('Randy', 'PERSON'), ('Patti', 'PERSON'), ('P..."
6,<16254169.1075863688286.JavaMail.evans@thyme>,phillip.allen@enron.com,"[david.l.johnson@enron.com, john.shafer@enron....",,2000-08-22 14:44:00+00:00,Please cc the following distribution list with...,Please cc the following distribution list with...,30,"[('Phillip Allen', 'PERSON'), ('Mike Grigsby',..."
11,<25459584.1075855687536.JavaMail.evans@thyme>,phillip.allen@enron.com,[stagecoachmama@hotmail.com],,2000-10-13 13:45:00+00:00,"Lucy,\n\n Here are the rentrolls:\n\n\n\n Open...","Lucy, Here are the rentrolls: Open them and sa...",54,"[('Lucy', 'PERSON'), ('1.', 'CARDINAL'), ('Sav..."
14,<2465689.1075855687605.JavaMail.evans@thyme>,phillip.allen@enron.com,[david.delainey@enron.com],,2000-10-05 13:26:00+00:00,"Dave, \n\n Here are the names of the west desk...","Dave, Here are the names of the west desk memb...",19,"[('Dave', 'PERSON'), ('Phillip', 'PERSON')]"


In [ ]:
# Reload the original dataset to ensure 'message_id' is present
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("corbt/enron-emails")
df = pd.DataFrame(dataset['train'])

# Re-apply cleaning
import re
def clean_email(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df['clean_message'] = df['body'].apply(clean_email)

# Re-apply transformation (word count)
df['word_count'] = df['clean_message'].apply(lambda x: len(x.split()))

# Re-apply enrichment (NER on sample)
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text[:1000])
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    return str(ents)

sample_df = df.head(2000).copy()
sample_df['entities'] = sample_df['clean_message'].apply(extract_entities)

df['entities'] = ""
df.loc[sample_df.index, 'entities'] = sample_df['entities']

# Now, re-run the normalization step
df = df[['message_id', 'from', 'to', 'subject', 'date', 'body',
         'clean_message', 'word_count', 'entities']]
df.head()


,message_id,from,to,subject,date,body,clean_message,word_count,entities
0,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,[tim.belden@enron.com],,2001-05-14 23:39:00+00:00,Here is our forecast\n\n,Here is our forecast,4,[]
1,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,[john.lavorato@enron.com],Re:,2001-05-04 20:51:00+00:00,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...,139,"[('Austin', 'PERSON')]"
2,<24216240.1075855687451.JavaMail.evans@thyme>,phillip.allen@enron.com,[leah.arsdall@enron.com],Re: test,2000-10-18 10:00:00+00:00,test successful. way to go!!!,test successful. way to go!!!,5,[]
3,<13505866.1075863688222.JavaMail.evans@thyme>,phillip.allen@enron.com,[randall.gay@enron.com],,2000-10-23 13:13:00+00:00,"Randy,\n\n Can you send me a schedule of the s...","Randy, Can you send me a schedule of the salar...",34,"[('Randy', 'PERSON'), ('Patti', 'PERSON'), ('P..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,phillip.allen@enron.com,[greg.piper@enron.com],Re: Hello,2000-08-31 12:07:00+00:00,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.,6,"[('Tuesday', 'DATE'), ('11:45', 'CARDINAL')]"


Save Triples to CSV

In [ ]:
triples_df.to_csv("kg_triples.csv", index=False)


Download


In [ ]:
from google.colab import files
files.download("kg_triples.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import List, Optional

@dataclass
class Email:
    message_id: str
    sender: str
    recipients: List[str] = field(default_factory=list)
    subject: Optional[str] = None
    date: Optional[datetime] = None
    body: Optional[str] = None
    clean_message: Optional[str] = None
    word_count: Optional[int] = None
    entities: List[tuple[str, str]] = field(default_factory=list)

# Example 1: Simple email
email1 = Email(
    message_id="<email1.123@example.com>",
    sender="alice@example.com",
    recipients=["bob@example.com", "charlie@example.com"],
    subject="Meeting Tomorrow",
    date=datetime(2023, 10, 26, 9, 0, 0),
    body="Hi Bob and Charlie, don't forget our meeting tomorrow at 10 AM.",
    clean_message="Hi Bob and Charlie, don't forget our meeting tomorrow at 10 AM.",
    word_count=14,
    entities=[('Bob', 'PERSON'), ('Charlie', 'PERSON'), ('tomorrow', 'DATE'), ('10 AM', 'TIME')]
)

# Example 2: Email with more details and an empty subject
email2 = Email(
    message_id="<email2.456@example.com>",
    sender="diana@example.com",
    recipients=["eve@example.com"],
    subject=None,
    date=datetime(2023, 10, 25, 14, 30, 0),
    body="I've attached the report you requested. Please let me know if you have any questions.",
    clean_message="I've attached the report you requested. Please let me know if you have any questions.",
    word_count=17,
    entities=[]
)

print("--- Email 1 ---")
print(email1)
print("\n--- Email 2 ---")
print(email2)


--- Email 1 ---
Email(message_id='<email1.123@example.com>', sender='alice@example.com', recipients=['bob@example.com', 'charlie@example.com'], subject='Meeting Tomorrow', date=datetime.datetime(2023, 10, 26, 9, 0), body="Hi Bob and Charlie, don't forget our meeting tomorrow at 10 AM.", clean_message="Hi Bob and Charlie, don't forget our meeting tomorrow at 10 AM.", word_count=14, entities=[('Bob', 'PERSON'), ('Charlie', 'PERSON'), ('tomorrow', 'DATE'), ('10 AM', 'TIME')])

--- Email 2 ---
Email(message_id='<email2.456@example.com>', sender='diana@example.com', recipients=['eve@example.com'], subject=None, date=datetime.datetime(2023, 10, 25, 14, 30), body="I've attached the report you requested. Please let me know if you have any questions.", clean_message="I've attached the report you requested. Please let me know if you have any questions.", word_count=17, entities=[])


In [ ]:
check_df.head()

,message_id,from,to,subject,date,body,clean_message,word_count,entities
0,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,['tim.belden@enron.com'],NaN,2001-05-14 23:39:00+00:00,Here is our forecast\n\n,Here is our forecast,4,[]
1,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,['john.lavorato@enron.com'],Re:,2001-05-04 20:51:00+00:00,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...,139,"[('Austin', 'PERSON')]"
2,<24216240.1075855687451.JavaMail.evans@thyme>,phillip.allen@enron.com,['leah.arsdall@enron.com'],Re: test,2000-10-18 10:00:00+00:00,test successful. way to go!!!,test successful. way to go!!!,5,[]
3,<13505866.1075863688222.JavaMail.evans@thyme>,phillip.allen@enron.com,['randall.gay@enron.com'],NaN,2000-10-23 13:13:00+00:00,"Randy,\n\n Can you send me a schedule of the s...","Randy, Can you send me a schedule of the salar...",34,"[('Randy', 'PERSON'), ('Patti', 'PERSON'), ('P..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,phillip.allen@enron.com,['greg.piper@enron.com'],Re: Hello,2000-08-31 12:07:00+00:00,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.,6,"[('Tuesday', 'DATE'), ('11:45', 'CARDINAL')]"
